In [1]:
library(readxl)
library(writexl)
library(ggplot2)
library(dplyr)
library(ggExtra)
library(anthroplus)
library(scales)
library(knitr)
library(officer)
library(flextable)
library(janitor)


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Attaching package: ‘officer’

The following object is masked from ‘package:readxl’:

    read_xlsx


Attaching package: ‘janitor’

The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test



In [2]:
male_data <- read_excel("male_data.xlsx")

# A. Analisis Status Gizi Menggunakan `anthropolus`
### 1. Menyiapkan Data

In [3]:
male_data_clean <- male_data %>%
  filter(
    !is.na(bb),
    !is.na(tb),
    !is.na(umur)
  ) %>%
  mutate(
    bb = as.numeric(bb),
    tb = as.numeric(tb),

    # age (years -> months)
    umur_bulan = as.numeric(umur) * 12,

    # BMI
    bmi = bb / (tb/100)^2
  )

# Raw data > Remove missing values > Convert age to months > Calculate BMI

### 2. Menghitung Z-Scores WHO BMI-for-age

In [4]:
who_results <- anthroplus_zscores(
  sex = rep(1, nrow(male_data_clean)),
  age_in_months = male_data_clean$umur_bulan,
  height_in_cm = male_data_clean$tb,
  weight_in_kg = male_data_clean$bb
)
# Age
# Height
# Weight
#      ↓
# WHO LMS reference
#      ↓
# WHO BMI-for-age z-score

### 3. Mengkategorikan Status Gizi

In [5]:
male_data_final <- male_data_clean %>%
  mutate(
    bmi_zscore_who = who_results$zbfa
  ) %>%
  mutate(
    bmi_category_who = case_when(

      bmi_zscore_who < -3 ~
        "Gizi Buruk",

      bmi_zscore_who >= -3 &
        bmi_zscore_who < -2 ~
        "Gizi Kurang",

      bmi_zscore_who >= -2 &
        bmi_zscore_who <= 1 ~
        "Gizi Normal",

      bmi_zscore_who > 1 &
        bmi_zscore_who <= 2 ~
        "Gizi Lebih",

      bmi_zscore_who > 2 ~
        "Obesitas"
    ),

    bmi_category_who = factor(
      bmi_category_who,
      levels = c(
        "Gizi Buruk",
        "Gizi Kurang",
        "Gizi Normal",
        "Gizi Lebih",
        "Obesitas"
      )
    )
  )
# WHO z-score
    #  ↓
# WHO cutoff
    #  ↓
# Nutritional status
male_data_final

# A tibble: 827 × 11
   ks    tl                   umur sex      tb    bb    bf   bmi umur_bulan
   <chr> <dttm>              <dbl> <chr> <dbl> <dbl> <dbl> <dbl>      <dbl>
 1 SM71  2013-06-24 00:00:00  12.3 M      143   39.5  13.4  19.3       148.
 2 SM72  2012-10-01 00:00:00  13   M      148.  37.5  11.1  17.2       156 
 3 SM73  2012-12-22 00:00:00  12.8 M      158   41.8   8.1  16.7       154.
 4 SM74  2013-02-14 00:00:00  12.6 M      160   64.7  22    25.3       151.
 5 SM75  2013-04-30 00:00:00  12.4 M      145   45.3   0    21.5       149.
 6 SM76  2012-10-28 00:00:00  12.9 M      166.  69.5  27.8  25.1       155.
 7 SM77  2012-10-10 00:00:00  12.9 M      148   42.3  11.4  19.3       155.
 8 SM78  2013-03-27 00:00:00  12.5 M      156.  44.4  12.2  18.4       150 
 9 SM79  2013-02-02 00:00:00  12.7 M      148.  51.2  23.4  23.5       152.
10 SM710 2012-05-20 00:00:00  13.3 M      160.  47.5  11.1  18.7       160.
# ℹ 817 more rows
# ℹ 2 more variables: bmi_zscore_who <dbl>, bmi_c

---
## PERHITUNGAN Z-SCORE 3 METODE: MANUAL, INTERPOLASI DAN PAKET anthropolus

In [6]:
# 1. CLEAN DATA
male_data_clean <- male_data %>%
  filter(
    !is.na(bb),
    !is.na(tb),
    !is.na(umur)
  ) %>%
  mutate(

    bb = as.numeric(bb),

    tb = as.numeric(tb),

    umur_bulan =
      as.numeric(umur) * 12,

    bmi =
      bb / (tb / 100)^2

  )

# =====================================================
# 2. WHO LMS TABLE
# =====================================================

who_lms <- anthroplus:::bfa_growth_standards %>%
  filter(sex == 1) %>%
  arrange(age)

# =====================================================
# 3. MANUAL WHO TABLE LOOKUP
# =====================================================

male_compare <- male_data_clean %>%

  mutate(

    age_months_who =
      round(umur_bulan)

  ) %>%

  left_join(

    who_lms %>%
      select(
        age,
        l,
        m,
        s
      ),

    by = c(
      "age_months_who" = "age"
    )

  ) %>%

  rename(

    l_manual = l,
    m_manual = m,
    s_manual = s

  ) 

# =====================================================
# 4. INTERPOLATED LMS
# =====================================================

male_compare <- male_compare %>%

  mutate(

    l_interp = approx(
      x = who_lms$age,
      y = who_lms$l,
      xout = umur_bulan
    )$y,

    m_interp = approx(
      x = who_lms$age,
      y = who_lms$m,
      xout = umur_bulan
    )$y,

    s_interp = approx(
      x = who_lms$age,
      y = who_lms$s,
      xout = umur_bulan
    )$y

  )

# =====================================================
# 5. MANUAL Z SCORE
# =====================================================

male_compare <- male_compare %>%

  mutate(

    bmi_z_manual = ifelse(

      l_manual == 0,

      log(
        bmi / m_manual
      ) / s_manual,

      (
        (bmi / m_manual)^l_manual - 1
      ) /
        (
          l_manual * s_manual
        )

    )

  )

# =====================================================
# 6. INTERPOLATED Z SCORE
# =====================================================

male_compare <- male_compare %>%

  mutate(

    bmi_z_interp = ifelse(

      l_interp == 0,

      log(
        bmi / m_interp
      ) / s_interp,

      (
        (bmi / m_interp)^l_interp - 1
      ) /
        (
          l_interp * s_interp
        )

    )

  )

# =====================================================
# 7. PACKAGE RESULT
# =====================================================

who_pkg <- anthroplus_zscores(

  sex = rep(
    2,
    nrow(male_compare)
  ),

  age_in_months =
    male_compare$umur_bulan,

  height_in_cm =
    male_compare$tb,

  weight_in_kg =
    male_compare$bb

)

male_compare$bmi_z_package <-
  who_pkg$zbfa

# =====================================================
# 8. DIFFERENCES
# =====================================================

male_compare <- male_compare %>%

  mutate(

    diff_l =
      l_interp - l_manual,

    diff_m =
      m_interp - m_manual,

    diff_s =
      s_interp - s_manual,

    diff_manual_pkg =
      bmi_z_manual -
      bmi_z_package,

    diff_interp_pkg =
      bmi_z_interp -
      bmi_z_package

  )

# =====================================================
# 9. FINAL TABLE
# =====================================================

comparison_full <- male_compare %>%

  select(

    umur_bulan,
    age_months_who,

    bmi,

    # LMS WHO Table
    l_manual,
    m_manual,
    s_manual,

    # LMS Interpolated
    l_interp,
    m_interp,
    s_interp,

    # LMS Difference
    diff_l,
    diff_m,
    diff_s,

    # Z Scores
    bmi_z_manual,
    bmi_z_interp,
    bmi_z_package,

    # Z-score Difference
    diff_manual_pkg,
    diff_interp_pkg

  ) %>%

  mutate(

    across(
      where(is.numeric),
      ~ round(.x, 6)
    )

  )

# =====================================================
# 10. DISPLAY
# =====================================================

# print(kable(comparison_full, digits = 6))

# =====================================================
# 11. VALIDATION METRICS
# =====================================================

metrics_block <- function(df, pred, ref, label = "") {

  x <- df[[pred]]
  y <- df[[ref]]

  cat("\n=====", label, "=====\n")

  cat("Correlation : ",
      round(cor(x, y, use = "complete.obs"), 6), "\n")

  cat("MAE         : ",
      round(mean(abs(x - y), na.rm = TRUE), 6), "\n")

  cat("RMSE        : ",
      round(sqrt(mean((x - y)^2, na.rm = TRUE)), 6), "\n")

  cat("Max Error   : ",
      round(max(abs(x - y), na.rm = TRUE), 6), "\n")
}

In [7]:
print(kable(comparison_full, digits = 6))
# comparison_full




| umur_bulan| age_months_who|      bmi| l_manual| m_manual| s_manual| l_interp| m_interp| s_interp|   diff_l|   diff_m|    diff_s| bmi_z_manual| bmi_z_interp| bmi_z_package| diff_manual_pkg| diff_interp_pkg|
|----------:|--------------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|---------:|------------:|------------:|-------------:|---------------:|---------------:|
|      147.6|            148| 19.31635|  -1.7604|  17.7551|  0.11656| -1.76204| 17.73246| 0.116428| -0.00164| -0.02264| -0.000132|     0.671972|     0.682124|          0.42|        0.251972|        0.262124|
|      156.0|            156| 17.23643|  -1.7168|  18.2330|  0.11898| -1.71680| 18.23300| 0.118980|  0.00000|  0.00000|  0.000000|    -0.495962|    -0.495962|         -0.68|        0.184038|        0.184038|
|      153.6|            154| 16.74411|  -1.7293|  18.1096|  0.11841| -1.73166| 18.08528| 0.118290| -0.00236| -0.02432| -0.000120|    -0.709045|    -0.696835|        

In [8]:
metrics_block(comparison_full, "bmi_z_manual", "bmi_z_package", "MANUAL vs PACKAGE")
metrics_block(comparison_full, "bmi_z_interp", "bmi_z_package", "INTERPOLATED vs PACKAGE")


===== MANUAL vs PACKAGE =====
Correlation :  0.994361 
MAE         :  0.177933 
RMSE        :  0.225436 
Max Error   :  1.952967 

===== INTERPOLATED vs PACKAGE =====
Correlation :  0.994347 
MAE         :  0.178377 
RMSE        :  0.225741 
Max Error   :  1.945212 


## KATEGORISASI

In [9]:
categorize_bmi_z <- function(z){

  case_when(

    z < -3 ~ "Gizi Buruk",

    z >= -3 & z < -2 ~
      "Gizi Kurang",

    z >= -2 & z <= 1 ~
      "Gizi Normal",

    z > 1 & z <= 2 ~
      "Gizi Lebih",

    z > 2 ~
      "Obesitas",

    TRUE ~ NA_character_

  )

}

comparison_full <- comparison_full %>%

  mutate(

    cat_manual =
      categorize_bmi_z(
        bmi_z_manual
      ),

    cat_interp =
      categorize_bmi_z(
        bmi_z_interp
      ),

    cat_package =
      categorize_bmi_z(
        bmi_z_package
      )

  )
comparison_full

# A tibble: 827 × 20
   umur_bulan age_months_who   bmi l_manual m_manual s_manual l_interp m_interp
        <dbl>          <dbl> <dbl>    <dbl>    <dbl>    <dbl>    <dbl>    <dbl>
 1       148.            148  19.3    -1.76     17.8    0.117    -1.76     17.7
 2       156             156  17.2    -1.72     18.2    0.119    -1.72     18.2
 3       154.            154  16.7    -1.73     18.1    0.118    -1.73     18.1
 4       151.            151  25.3    -1.75     17.9    0.118    -1.75     17.9
 5       149.            149  21.5    -1.76     17.8    0.117    -1.76     17.8
 6       155.            155  25.1    -1.72     18.2    0.119    -1.72     18.2
 7       155.            155  19.3    -1.72     18.2    0.119    -1.72     18.2
 8       150             150  18.4    -1.75     17.9    0.117    -1.75     17.9
 9       152.            152  23.5    -1.74     18.0    0.118    -1.74     18.0
10       160.            160  18.7    -1.69     18.5    0.120    -1.69     18.5
# ℹ 817 more rows
#

In [10]:
write_xlsx(comparison_full, "male_nutritional_status.xlsx")

---
## RINGKASAN STATISTIK

In [11]:
create_category_summary <- function(
    data,
    category_var,
    digits = 1
){

  result <- data %>%

    count(
      Category = .data[[category_var]]
    ) %>%

    mutate(

      Percent =
        round(
          100 * n / sum(n),
          digits
        ),

      n_percent =
        paste0(
          n,
          " (",
          Percent,
          "%)"
        )

    ) %>%

    rename(
      N = n
    )

  total_row <- tibble::tibble(

    Category = "Total",

    N = sum(result$N),

    Percent = 100,

    n_percent =
      paste0(
        sum(result$N),
        " (100%)"
      )

  )

  bind_rows(
    result,
    total_row
  )

}

In [12]:
create_category_summary(
  comparison_full,
  "cat_package"
)

# A tibble: 7 × 4
  Category        N Percent n_percent  
  <chr>       <int>   <dbl> <chr>      
1 Gizi Buruk     11     1.3 11 (1.3%)  
2 Gizi Kurang    76     9.2 76 (9.2%)  
3 Gizi Lebih    107    12.9 107 (12.9%)
4 Gizi Normal   569    68.8 569 (68.8%)
5 Obesitas       58     7   58 (7%)    
6 NA              6     0.7 6 (0.7%)   
7 Total         827   100   827 (100%) 

In [13]:
summarize_single_variable <- function(
    female_data_final,
    var,
    digits = 3
){

  x <- female_data_final[[var]]

  tibble(

    Statistic = c(
      "N",
      "Mean ± SD",
      "Median (IQR)",
      "Minimum",
      "Maximum"
    ),

    Value = c(

      sum(!is.na(x)),

      paste0(
        round(
          mean(x, na.rm = TRUE),
          digits
        ),
        " ± ",
        round(
          sd(x, na.rm = TRUE),
          digits
        )
      ),

      paste0(
        round(
          median(x, na.rm = TRUE),
          digits
        ),
        " (",
        round(
          quantile(
            x,
            0.25,
            na.rm = TRUE
          ),
          digits
        ),
        ", ",
        round(
          quantile(
            x,
            0.75,
            na.rm = TRUE
          ),
          digits
        ),
        ")"
      ),

      round(
        min(x, na.rm = TRUE),
        digits
      ),

      round(
        max(x, na.rm = TRUE),
        digits
      )

    )

  )

}

In [14]:

summarize_single_variable(
  comparison_full,
  "bmi_z_package"
)


# A tibble: 5 × 2
  Statistic    Value             
  <chr>        <chr>             
1 N            821               
2 Mean ± SD    -0.28 ± 1.433     
3 Median (IQR) -0.4 (-1.31, 0.66)
4 Minimum      -5.05             
5 Maximum      3.91              